# Native engine runs with tracking

Run the single native engine from one schema-3 JSON file and keep a durable,
comparable record of every run. The default input is the invented
[feature example](../../examples/design/native_feature_options.proposed.json);
it is not a calibration or a biological claim. See
[native execution](../../docs/NATIVE_EXECUTION.md) and the
[implementation report](../../docs/research/NATIVE_ENGINE_IMPLEMENTATION_REPORT.md).

Every run uses the same path as the CLI and API: strict JSON parsing, one
resolver, one engine lifecycle and one exporter. Each run folder under
`.runtime/notebook-native-engine/` (not version controlled) contains:

| File | Purpose |
|---|---|
| `parameters.json` | The exact input used, after any edit below; reload it with the CLI |
| `resolved_config.json` | Active laws per order, inactive settings, units, RNG streams, hashes |
| `run_record.json` | Label, time, input file hash, identities, counts, output hashes, git state |
| `run_manifest.json` | Environment, dependency versions, output hashes, model identities |
| `nodes.csv`, `edges.csv`, `organs.csv` | Root graph and organ records |

Three identities are recorded. The **supplied input** hash changes with any
input byte, including disabled settings. The **effective model** hash changes
only with values that can affect the model; labels, limits and disabled
settings never change it. The **run identity** also binds the seed, the
generator source and the numeric environment. Every run is appended to
`ledger.jsonl`, so later sessions can find and compare earlier runs.

In [ ]:
from pathlib import Path
import datetime
import json

import pandas as pd
from IPython.display import display

REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "deeprootgen").is_dir())
CONFIG = REPO / "examples/design/native_feature_options.proposed.json"  # invented example
SESSION = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT = REPO / ".runtime/notebook-native-engine" / SESSION
LEDGER = REPO / ".runtime/notebook-native-engine/ledger.jsonl"
RUN = True  # Set False to inspect the resolved configuration without running.

## 1. Inspect the resolved configuration before running

Nothing runs here. The table shows the law chosen for each stage and order;
a replaced native fallback is named in the explanation. Unknown keys, duplicate
keys or an unsupported enabled feature fail at this step.

In [ ]:
from deeprootgen.data_model import explain_config

description = explain_config(CONFIG)
laws = pd.DataFrame(
    {
        order: {stage: law if isinstance(law, str) else law["law"] for stage, law in entry.items()}
        for order, entry in description["laws"]["by_order"].items()
    }
).T
display(laws)
print("Inactive feature blocks kept in the file:", sorted(description["features"]["inactive"]))
print(json.dumps(description["hashes"], indent=2))

## 2. Run primary curvature off, on, then off from the same file

Only `features.curvature.enabled` changes between runs. The edited input is
saved as `parameters.json` in each run folder.

In [ ]:
from deeprootgen.run_records import read_ledger, tracked_run


def curvature(enabled):
    def edit(spec):
        spec["features"]["curvature"]["enabled"] = enabled

    return edit


records = []
if RUN:
    for label, enabled in (("curvature-off-1", False), ("curvature-on", True), ("curvature-off-2", False)):
        records.append(
            tracked_run(CONFIG, OUTPUT / label, label=label, ledger=LEDGER, edit=curvature(enabled))
        )
    display(
        pd.DataFrame(
            [
                {
                    "label": r["label"],
                    "nodes": r["node_count"],
                    "active features": ", ".join(r["active_features"]) or "-",
                    "effective model": r["model_identity"]["effective_model_sha256"][:12],
                    "supplied input": r["model_identity"]["supplied_input_sha256"][:12],
                    "nodes.csv": r["outputs"]["nodes.csv"][:12],
                }
                for r in records
            ]
        )
    )
else:
    print("Not run. Set RUN = True.")

## 3. Check what must hold

Both "off" runs must be byte-identical with identical identities: switching a
feature off restores the native baseline exactly. The "on" run keeps the same
organs, sampled lengths and mass; only primary centrelines (and their extra
corner nodes) change.

In [ ]:
if RUN:
    off_1, on, off_2 = records
    assert off_1["outputs"]["nodes.csv"] == off_2["outputs"]["nodes.csv"]
    assert off_1["model_identity"] == off_2["model_identity"]
    assert on["model_identity"]["effective_model_sha256"] != off_1["model_identity"]["effective_model_sha256"]
    organs = {r["label"]: pd.read_csv(Path(r["outdir"]) / "organs.csv") for r in records}
    keys = ["organ_id", "order", "parent_organ_id"]
    assert organs["curvature-off-1"][keys].equals(organs["curvature-on"][keys])
    mass = organs["curvature-off-1"].mass_g
    print("Largest relative organ mass difference:", float(((organs["curvature-on"].mass_g - mass).abs() / mass).max()))

## 4. Toggle another feature the same way

Every feature block works the same way. Here point taper replaces the cylinder
material. Organs, lengths and geometry stay the same; segment diameters, the
fine/structural class of each segment and organ mass follow the taper, which
conserves the integrated volume. Enabling an unsupported combination (for
example `parent_diameter_cap` without taper) fails in step 1 before running.

In [ ]:
def feature(name, enabled):
    def edit(spec):
        spec["features"][name]["enabled"] = enabled

    return edit


if RUN:
    taper = tracked_run(CONFIG, OUTPUT / "taper-on", label="taper-on", ledger=LEDGER, edit=feature("taper", True))
    organs_taper = pd.read_csv(Path(taper["outdir"]) / "organs.csv")
    assert organs_taper[keys].equals(organs["curvature-off-1"][keys])
    comparison = organs["curvature-off-1"][["organ_id", "order", "length_cm", "mass_g"]].merge(
        organs_taper[["organ_id", "material_law", "mass_g"]], on="organ_id", suffixes=(" cylinder", " taper")
    )
    display(comparison)

## 5. Find, compare and reproduce runs

The ledger accumulates across sessions. To reproduce a run, rerun its saved
input and compare identities. The effective-model hash must match; the run
identity also matches when the code and numeric environment are unchanged.

```bash
python -m deeprootgen simulate --config <run folder>/parameters.json --outdir <new folder>
```

In [ ]:
ledger = pd.DataFrame(read_ledger(LEDGER))
columns = ["created_utc", "label", "seed", "node_count", "active_features", "outdir"]
display(ledger[columns].tail(10) if not ledger.empty else ledger)